## Embedding

In [ ]:
from nomic import embed
## get the user files
import pypdfium2 as pdfium
pdf = "C:\\Users\\andre\\Downloads\\designing-machine-learning-systems-an-iterative-process-for-production-ready-applications-1nbsped-1098107969-9781098107963.pdf"
pdf_object = pdfium.PdfDocument(pdf)
text = "\n".join(
    p.get_textpage().get_text_range() 
    for p in pdf_object
)

##chunking Strategy
n = len(text)
chunk_overlap = 150
chunk_size = 3000
chunks = [text[i*chunk_size-chunk_overlap:(i+1)*chunk_size]  for i in range(1,n//chunk_size)]
chunks.append(text[0:chunk_size])

##embed the chunks
nomic_res = embed.text(chunks,model="nomic-embed-text-v1.5",task_type='search_document',
    dimensionality=256,)

embeddings = nomic_res["embeddings"]



## Create the search index

In [ ]:
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
# 1. Connect with directConnection=True for your local Atlas container
client = MongoClient("mongodb://localhost:27017/?directConnection=true")
db = client["genai_db"]
collection_name = "knowledgestore"

if collection_name not in db.list_collection_names():
    db.create_collection(collection_name)

knowledge_store_collection = db[collection_name]

## defining the search index model
search_index_model = SearchIndexModel(
    definition= {
        "fields":[
            {"type":"vector",
             "path":"embeddings",
             "numDimensions":256,
             "similarity":"cosine",
             "quantization":"scalar"}
        ]
    },
    name="vector-index",
    type="vectorSearch"
)

import time
# 4. Create the index
try:
    result = knowledge_store_collection.create_search_index(model=search_index_model)
    print(f"Index creation request submitted: {result}")
    
    # 5. Optional: Wait for the index to be ready
    print("Waiting for index to build...")
    while True:
        indices = list(knowledge_store_collection.list_search_indexes(name="vector-index"))
        if indices and indices[0].get("queryable"):
            print("Index is now queryable!")
            break
        time.sleep(2)
        
except Exception as e:
    print(f"Error creating index: {e}")

Index creation request submitted: vector-index
Waiting for index to build...
Index is now queryable!


In [108]:
list(knowledge_store_collection.list_indexes())

[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])]

In [109]:
knowledge_store_collection.create_index(
    "expireAt",
    expireAfterSeconds=0
)

'expireAt_1'

In [112]:
from datetime import timezone,timedelta,datetime

data = {"text":chunks[0],"embeddings":embeddings[0],"userid":1,"expireAt":datetime.now(timezone.utc) + timedelta(seconds=5)}

In [114]:
datetime.now(timezone.utc) + timedelta(seconds=5)

datetime.datetime(2026, 4, 6, 15, 14, 38, 844640, tzinfo=datetime.timezone.utc)

In [113]:
knowledge_store_collection.insert_one(data)

InsertOneResult(ObjectId('69d3cdc71b74726b4f2af70b'), acknowledged=True)

### Inserting data into the index

In [92]:
from datetime import datetime
data_user_1= [{"text":chunks[i],"embeddings":embeddings[i],"userid":1,"createdAt":datetime.now()} for i in range(0,50)]
data_user_2 = [{"text":chunks[i],"embeddings":embeddings[i],"userid":2,"createdAt":datetime.now()} for i in range(40,279)]

In [104]:
len(data_user_1)

50

In [95]:
knowledge_store_collection.insert_many(data_user_1)

InsertManyResult([ObjectId('69d3c7cc1b74726b4f2af6d7'), ObjectId('69d3c7cc1b74726b4f2af6d8'), ObjectId('69d3c7cc1b74726b4f2af6d9'), ObjectId('69d3c7cc1b74726b4f2af6da'), ObjectId('69d3c7cc1b74726b4f2af6db'), ObjectId('69d3c7cc1b74726b4f2af6dc'), ObjectId('69d3c7cc1b74726b4f2af6dd'), ObjectId('69d3c7cc1b74726b4f2af6de'), ObjectId('69d3c7cc1b74726b4f2af6df'), ObjectId('69d3c7cc1b74726b4f2af6e0'), ObjectId('69d3c7cc1b74726b4f2af6e1'), ObjectId('69d3c7cc1b74726b4f2af6e2'), ObjectId('69d3c7cc1b74726b4f2af6e3'), ObjectId('69d3c7cc1b74726b4f2af6e4'), ObjectId('69d3c7cc1b74726b4f2af6e5'), ObjectId('69d3c7cc1b74726b4f2af6e6'), ObjectId('69d3c7cc1b74726b4f2af6e7'), ObjectId('69d3c7cc1b74726b4f2af6e8'), ObjectId('69d3c7cc1b74726b4f2af6e9'), ObjectId('69d3c7cc1b74726b4f2af6ea'), ObjectId('69d3c7cc1b74726b4f2af6eb'), ObjectId('69d3c7cc1b74726b4f2af6ec'), ObjectId('69d3c7cc1b74726b4f2af6ed'), ObjectId('69d3c7cc1b74726b4f2af6ee'), ObjectId('69d3c7cc1b74726b4f2af6ef'), ObjectId('69d3c7cc1b74726b4f2af6

## Retrieval

In [ ]:
query = "What is data engineering, can you explain the processes in data engineering"

embeddings = embed.text([query],model="nomic-embed-text-v1.5",task_type='search_query',
    dimensionality=256,)

qe = embeddings["embeddings"][0]


In [86]:
res = db[collection_name].aggregate([
    {
        "$vectorSearch":{
            "index":"vector-index",
            "path":"embeddings",
            "queryVector":qe,
            "numCandidates":100,
            "limit":5
        }
        },{       
            "$project": {
            "text": 1,
            "score": {"$meta": "vectorSearchScore"}
        }
    }

])

In [87]:
for doc in res:
   print(doc)

{'_id': ObjectId('69d3a7871b74726b4f2af5e7'), 'text': 'l continue to discuss data storage engines,\r\nalso known as databases, for the two major types of processing: transactional\r\nand analytical.\r\nWhen working with data in production, you usually work with data across\r\nmultiple processes and services. For example, you might have a feature\r\nengineering service that computes features from raw data, and a prediction\r\nservice to generate predictions based on computed features. This means that\r\nyou’ll have to pass computed features from the feature engineering service to the\r\nprediction service. In the following section of the chapter, we’ll discuss different\r\nmodes of data passing across processes.\r\nDuring the discussion of different modes of data passing, we’ll learn about two\ndistinct types of data: historical data in data storage engines, and streaming data\r\nin real-time transports. These two different types of data require different\r\nprocessing paradigms, which 